In [2]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-02-05 14:09:28 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Introducción

Por dinámica de tiempo se quiere decir que un cliente se puede vincular, luego desvincularse y finalmente volverse a vincular

Se evidencia que los registros en la tabla `resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf` las vinculaciones de adquirencia no comparte registros con la tabla `resultados_wompi.wompi_merchants` que contiene las vinculaciones a wompi

In [32]:
# HASTA QUE AÑO MES SE HAN ACTUALIZADO LAS TRXS
periodo_actual_trxs = '202511'
periodo_actual_trxs

'202511'

# Evolución vinculación aceptación comercios [Adquirencia + Wompi]

## Adquirencia

In [33]:
dict_ult_ing_adqu_vinc = helper.obtener_ultima_ingestion('resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf')
dict_ult_ing_adqu_vinc

2026-01-16 16:42:22 - [INFO] - Buscando fechas para resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2026-01-16 16:42:23 - [INFO] - Finalizo la busqueda, duracion: 00:00.4, resultado: {'year': 2026, 'month': 1, 'day': 7}


{'year': 2026, 'month': 1, 'day': 7}

In [34]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_hist_vinc_adqu PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_hist_vinc_adqu STORED AS PARQUET AS WITH vinculados AS
  (SELECT codigo_unico,
          extract(to_timestamp(concat(cast(YEAR AS string), '-', cast(MONTH AS string), '-01'), 'yyyy-M-dd') - INTERVAL 1 MONTH, 'year')*100 + extract(to_timestamp(concat(cast(YEAR AS string), '-', cast(MONTH AS string), '-01'), 'yyyy-M-dd') - INTERVAL 1 MONTH, 'month') AS fecha_ym
      FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
      WHERE YEAR <= """ + str(dict_ult_ing_adqu_vinc['year']) + """
      AND MONTH BETWEEN 1 AND 12
      AND DAY BETWEEN 1 AND 31),
                                                                                       conteo AS
  (SELECT fecha_ym,
          count(*) AS num_vinc_new,
          cast(left(cast(fecha_ym AS STRING), 4) AS int) AS YEAR,
          cast(right(cast(fecha_ym AS STRING), 2) AS int) AS mes
   FROM vinculados
   GROUP BY 1)
SELECT fecha_ym,
       num_vinc_new,
       sum(num_vinc_new) OVER (PARTITION BY YEAR
                           ORDER BY YEAR,
                                    mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_vinc_new_cumsum_ym,
                          'adquirencia' AS producto
FROM conteo
ORDER BY fecha_ym DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_hist_vinc_adqu;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 21/21      DROP ...o_aceptacion_comercios_hist_vinc_adqu   finalizado   04:42:23 PM     00:00.2 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 22/22    CREATE ...o_aceptacion_comercios_hist_vinc_adqu   finalizado   04:42:23 PM     00:01.1 
-------------------------------------------------------------------------------------------------
--------------------

## Wompi

In [35]:
dict_ult_ing_wompi_merch = helper.obtener_ultima_ingestion('resultados_wompi.wompi_merchants')
dict_ult_ing_wompi_merch


2026-01-16 16:42:41 - [INFO] - Buscando fechas para resultados_wompi.wompi_merchants
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2026-01-16 16:42:42 - [INFO] - Finalizo la busqueda, duracion: 00:00.4, resultado: {'year': 2026, 'month': 1, 'day': 16}


{'year': 2026, 'month': 1, 'day': 16}

In [36]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_hist_vinc_wompi PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_hist_vinc_wompi STORED AS PARQUET AS
WITH conteo AS (
SELECT CASt(left(cast(creado as string), 6) as int) as fecha_ym, 
       count(*) as num_vinc_new
FROM resultados_wompi.wompi_merchants
WHERE YEAR = """ + str(dict_ult_ing_wompi_merch['year']) + """
  AND MONTH = """ + str(dict_ult_ing_wompi_merch['month']) + """
  AND DAY = """ + str(dict_ult_ing_wompi_merch['day']) + """
  AND modelo = 'Agregador'
  and activo = 'A'
  and desembolsos_permitidos = 'Si'
GROUP BY 1
), conteo_y_m AS (
SELECT fecha_ym,
        num_vinc_new,
        cast(left(cast(fecha_ym AS STRING), 4) AS int) AS YEAR,
        cast(right(cast(fecha_ym AS STRING), 2) AS int) AS mes
FROM conteo
)
SELECT fecha_ym,
       num_vinc_new,
       sum(num_vinc_new) OVER (PARTITION BY YEAR
                           ORDER BY YEAR,
                                    mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_vinc_new_cumsum_ym,
        'wompi' AS producto
FROM conteo_y_m
ORDER BY fecha_ym DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_hist_vinc_wompi;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 24/24      DROP ..._aceptacion_comercios_hist_vinc_wompi   finalizado   04:42:42 PM     00:00.1 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 25/25    CREATE ..._aceptacion_comercios_hist_vinc_wompi   finalizado   04:42:42 PM     00:01.4 
-------------------------------------------------------------------------------------------------
--------------------

# Vinculación aceptación comercios

In [37]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_vinc PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_vinc STORED AS PARQUET AS
WITH junte AS
  (SELECT fecha_ym,
          num_vinc_new,
          num_vinc_new_cumsum_ym,
          producto
   FROM proceso.mdo_aceptacion_comercios_hist_vinc_adqu
   UNION ALL SELECT fecha_ym,
                    num_vinc_new,
                    num_vinc_new_cumsum_ym,
                    producto
   FROM proceso.mdo_aceptacion_comercios_hist_vinc_wompi),
     agregado AS
  (SELECT fecha_ym,
          cast(left(cast(fecha_ym AS STRING), 4) AS int) AS YEAR,
          cast(right(cast(fecha_ym AS STRING), 2) AS int) AS mes,
          sum(num_vinc_new) AS num_vinc_new,
          1 AS secuencia2
   FROM junte
   GROUP BY 1,
            2,
            3)
SELECT fecha_ym,
       num_vinc_new,
       sum(num_vinc_new) OVER (PARTITION BY secuencia2 ORDER BY fecha_ym ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_vinc_cumsum,
       sum(num_vinc_new) OVER (PARTITION BY YEAR
                           ORDER BY YEAR,
                                    mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_vinc_new_cumsum_ym
FROM agregado
ORDER BY fecha_ym DESC;
"""
df_outcome = helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_vinc;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 27/27      DROP    proceso.mdo_aceptacion_comercios_vinc   finalizado   04:42:44 PM     00:00.2 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 28/28    CREATE    proceso.mdo_aceptacion_comercios_vinc   finalizado   04:42:45 PM     00:02.2 
-------------------------------------------------------------------------------------------------
--------------------

In [38]:
sql = """
SELECT fecha_ym,
       num_vinc_new,
       num_vinc_cumsum,
       num_vinc_new_cumsum_ym
FROM proceso.mdo_aceptacion_comercios_vinc
ORDER BY fecha_ym DESC
"""
df_outcome = helper.obtener_dataframe(sql)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 30/30 DATAFRAME                                            ejecutando   04:42:48 PM             

2026-01-16 16:42:49 - [INFO] - 88 filas, 4 columnas, 00:00.5 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


 30/30 DATAFRAME                                            finalizado   04:42:48 PM     00:00.7 
-------------------------------------------------------------------------------------------------


In [39]:
df_outcome[40:].head(20)

,fecha_ym,num_vinc_new,num_vinc_cumsum,num_vinc_new_cumsum_ym
40,202209,5797,178187,49591
41,202208,6820,172390,43794
42,202207,7554,165570,36974
43,202206,6363,158016,29420
44,202205,7256,151653,23057
45,202204,2745,144397,15801
46,202203,3712,141652,13056
47,202202,4970,137940,9344
48,202201,4374,132970,4374
49,202112,5570,128596,115843


# Uso de vinculados nuevos aceptación comercios

In [40]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_num_vinc_new_uso PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_num_vinc_new_uso STORED AS PARQUET AS
WITH uso_adqui AS
  (SELECT periodo,
          count(*) AS num_vinc_new_uso_adqu
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist_con_dup
   WHERE producto = 'adqui'
   AND tipo_cliente = 'nuevos'
   GROUP BY 1),
     uso_wompi AS
  (SELECT periodo,
          count(*) AS num_vinc_new_uso_womp
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist_con_dup
   WHERE producto = 'wompi'
   AND tipo_cliente = 'nuevos'
   GROUP BY 1)
SELECT a.periodo as fecha_ym,
       a.num_vinc_new_uso_adqu + nvl(b.num_vinc_new_uso_womp, 0) AS num_vinc_new_uso_cumsum_ym
FROM uso_adqui AS a
LEFT JOIN uso_wompi AS b ON a.periodo = b.periodo
ORDER BY a.periodo DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_num_vinc_new_uso;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 31/31      DROP ...aceptacion_comercios_num_vinc_new_uso   finalizado   04:42:49 PM     00:00.1 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 32/32    CREATE ...aceptacion_comercios_num_vinc_new_uso   finalizado   04:42:50 PM     00:14.1 
-------------------------------------------------------------------------------------------------
--------------------

# Uso de vinculados viejos aceptación comercios

In [41]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_num_vinc_old_uso PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_num_vinc_old_uso STORED AS PARQUET AS
WITH uso_adqui AS
  (SELECT periodo,
          count(*) AS num_vinc_old_uso_adqu
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist_con_dup
   WHERE producto = 'adqui'
   AND tipo_cliente = 'viejos'
   GROUP BY 1),
     uso_wompi AS
  (SELECT periodo,
          count(*) AS num_vinc_old_uso_womp
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist_con_dup
   WHERE producto = 'wompi'
   AND tipo_cliente = 'viejos'
   GROUP BY 1)
SELECT a.periodo as fecha_ym,
       a.num_vinc_old_uso_adqu + nvl(b.num_vinc_old_uso_womp, 0) AS num_vinc_old_uso_cumsum_ym
FROM uso_adqui AS a
LEFT JOIN uso_wompi AS b ON a.periodo = b.periodo
ORDER BY a.periodo DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_num_vinc_old_uso;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 34/34      DROP ...aceptacion_comercios_num_vinc_old_uso   finalizado   04:43:06 PM     00:00.2 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 35/35    CREATE ...aceptacion_comercios_num_vinc_old_uso   finalizado   04:43:06 PM     00:10.7 
-------------------------------------------------------------------------------------------------
--------------------

# Uso de vinculados todos aceptación comercios

In [42]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_num_vinc_all_uso PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_num_vinc_all_uso STORED AS PARQUET AS
WITH uso_adqui AS
  (SELECT periodo,
          count(*) AS num_vinc_all_uso_adqu
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist_con_dup
   WHERE producto = 'adqui'
   AND tipo_cliente = 'todos'
   GROUP BY 1),
     uso_wompi AS
  (SELECT periodo,
          count(*) AS num_vinc_all_uso_womp
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist_con_dup
   WHERE producto = 'wompi'
   AND tipo_cliente = 'todos'
   GROUP BY 1)
SELECT a.periodo as fecha_ym,
       a.num_vinc_all_uso_adqu + nvl(b.num_vinc_all_uso_womp, 0) AS num_vinc_all_uso_cumsum_ym
FROM uso_adqui AS a
LEFT JOIN uso_wompi AS b ON a.periodo = b.periodo
ORDER BY a.periodo DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_num_vinc_all_uso;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 37/37      DROP ...aceptacion_comercios_num_vinc_all_uso   finalizado   04:43:19 PM     00:00.2 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 38/38    CREATE ...aceptacion_comercios_num_vinc_all_uso   finalizado   04:43:20 PM     00:12.9 
-------------------------------------------------------------------------------------------------
--------------------

# Tabla resultado

In [43]:
sql = """
WITH outcome1 AS
  (SELECT a.fecha_ym,
          a.num_vinc_new,
          a.num_vinc_cumsum,
          a.num_vinc_new_cumsum_ym,
          nvl(b.num_vinc_new_uso_cumsum_ym, 0) AS num_vinc_new_uso_cumsum_ym,
          round(nvl(b.num_vinc_new_uso_cumsum_ym, 0)/a.num_vinc_new_cumsum_ym, 4) AS num_vinc_new_prop_uso,
          nvl(c.num_vinc_old_uso_cumsum_ym, 0) AS num_vinc_old_uso_cumsum_ym,
          nvl(d.num_vinc_all_uso_cumsum_ym, 0) AS num_vinc_all_uso_cumsum_ym,
          round(nvl(d.num_vinc_all_uso_cumsum_ym, 0)/a.num_vinc_cumsum, 4) AS num_vinc_all_prop_uso,
          left(cast(a.fecha_ym AS string), 4) AS YEAR,
          right(cast(a.fecha_ym AS string), 2) AS mes
   FROM proceso.mdo_aceptacion_comercios_vinc AS a
   LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_new_uso AS b ON a.fecha_ym = b.fecha_ym
   LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_old_uso AS c ON a.fecha_ym = c.fecha_ym
   LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_all_uso AS d ON a.fecha_ym = d.fecha_ym),
     outcome2 AS
  (SELECT fecha_ym,
          num_vinc_cumsum AS num_vinc_old,
          cast(cast(YEAR AS int) + 1 AS string) AS YEAR
   FROM outcome1
   WHERE mes = '12')
SELECT a.fecha_ym,
       CONCAT(a.YEAR, '/', a.mes, '/', '01') AS fecha_ym2,
       a.num_vinc_new,
       a.num_vinc_new_cumsum_ym,
       a.num_vinc_new_uso_cumsum_ym,
       a.num_vinc_new_prop_uso,
       b.num_vinc_old,
       a.num_vinc_old_uso_cumsum_ym,
       round(a.num_vinc_old_uso_cumsum_ym/b.num_vinc_old, 4) AS num_vinc_old_prop_uso,
       a.num_vinc_cumsum,
       a.num_vinc_all_uso_cumsum_ym,
       a.num_vinc_all_prop_uso
FROM outcome1 AS a
LEFT JOIN outcome2 AS b ON a.year = b.year
WHERE a.fecha_ym BETWEEN 202201 AND 202511
ORDER BY a.fecha_ym DESC;
"""
# print(sql)
df_outcome = helper.obtener_dataframe(sql)
df_outcome

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 40/40 DATAFRAME                                            finalizado   04:43:34 PM     00:01.1 
-------------------------------------------------------------------------------------------------


2026-01-16 16:43:35 - [INFO] - 47 filas, 12 columnas, 00:00.9 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


,fecha_ym,fecha_ym2,num_vinc_new,num_vinc_new_cumsum_ym,num_vinc_new_uso_cumsum_ym,num_vinc_new_prop_uso,num_vinc_old,num_vinc_old_uso_cumsum_ym,num_vinc_old_prop_uso,num_vinc_cumsum,num_vinc_all_uso_cumsum_ym,num_vinc_all_prop_uso
0,202511,2025/11/01,15051,100796,38823,0.3852,323753,103274,0.3190,424549,142097,0.3347
1,202510,2025/10/01,16248,85745,34180,0.3986,323753,102731,0.3173,409498,136911,0.3343
2,202509,2025/09/01,22796,69497,29340,0.4222,323753,102090,0.3153,393250,131430,0.3342
3,202508,2025/08/01,7078,46701,22020,0.4715,323753,101310,0.3129,370454,123330,0.3329
4,202507,2025/07/01,8146,39623,19882,0.5018,323753,100437,0.3102,363376,120319,0.3311
5,202506,2025/06/01,6345,31477,15970,0.5074,323753,99408,0.3070,355230,115378,0.3248
6,202505,2025/05/01,6469,25132,12905,0.5135,323753,98241,0.3034,348885,111146,0.3186
7,202504,2025/04/01,5234,18663,9492,0.5086,323753,96434,0.2979,342416,105926,0.3093
8,202503,2025/03/01,4855,13429,6541,0.4871,323753,94424,0.2917,337182,100965,0.2994
9,202502,2025/02/01,4466,8574,3989,0.4652,323753,91308,0.2820,332327,95297,0.2868


In [44]:
df_outcome.to_excel('main_data/evolucion_vinculacion_y_uso_adquirencia_y_wompi_con_duplicados.xlsx')

# Eliminación tablas proceso.

In [3]:
# Eliminación tablas proceso.
tablas_borrar = ['proceso.mdo_aceptacion_comercios_hist_vinc_adqu', 'proceso.mdo_aceptacion_comercios_hist_vinc_wompi', 'proceso.mdo_aceptacion_comercios_vinc', 'proceso.mdo_aceptacion_comercios_num_vinc_new_uso', 'proceso.mdo_aceptacion_comercios_num_vinc_old_uso', 'proceso.mdo_aceptacion_comercios_num_vinc_all_uso']

for tabla in tablas_borrar:
    sql_drop = f"""DROP TABLE IF EXISTS {tabla} PURGE;"""
    helper.ejecutar_consulta(sql_drop)

2026-02-05 14:09:34 - [INFO] - Transcurrido: 1770318574, Tiempo de Refresco = 1000


------------------------------------------------------------------------------------------
  i  tipo                  nombre                     estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------
 1/1 DROP ...o_aceptacion_comercios_hist_vinc_adqu   finalizado   02:09:35 PM     00:00.7 
------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------
  i  tipo                  nombre                     estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------
 2/2 DROP ..._aceptacion_comercios_hist_vinc_wompi   finalizado   02:09:35 PM     00:00.1 
------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------